# 💧 USGS Water Level Prediction – Google Colab Training Demo

**Version:** 1.0 (Training Pipeline Only)  
**Model:** EfficientNet Regression (adapted from production pipeline)  
**Target environment:** Google Colab Free Tier · T4 GPU  

---

## 📁 Expected Folder Structure
```
/content/water_level_demo/
├── data/
│   └── images/          ← your site images go here (or in a Drive folder)
│   └── labels.csv       ← your CSV label file
├── results/             ← all training outputs appear here
├── train_demo.py        ← training module (auto-copied from this repo)
├── data_acquisition.py  ← data pipeline  (auto-copied from this repo)\n",
├── app.py               ← Gradio UI    (auto-copied from this repo)
└── Water_Level_Training_Demo.ipynb  ← this notebook
```

## 🗂️ CSV Format
Your CSV needs **at least two columns**:
| Column purpose | Accepted names |
|---|---|
| Image filename/path | `image_path`, `dfile_path`, `filename`, `file`, `image` |
| Water level value | `water_level`, `usgstrue_wl`, `gage_height`, `target`, `label` |
| Timestamp (optional) | `dt_pdatetime`, `timestamp`, `datetime`, `date_time` |

## 🚀 Quick Start
1. **Run Cell 0** – checks GPU and sets up folder structure  
2. **Run Cell 1** – installs packages (takes ~2 min on first run)  
3. **Run Cell 2** – copies `train_demo.py` and `app.py` to `/content/water_level_demo/`  
4. **Run Cell 3** – launches the Gradio UI (opens a public share link)  
5. Use the Gradio UI to upload data, configure, and train!

## ⚠️ If Colab Disconnects
- Re-run all cells from the top (Cells 0 → 3).  
- Your results in `/content/water_level_demo/results/` will be lost unless you enabled **'Copy outputs to Google Drive'** in the UI.  
- **Tip:** Mount Google Drive in Cell 0 and always enable the Drive save option.

## 💾 Memory Tips (T4 · 15 GB VRAM)
| Config | VRAM usage | Speed |
|---|---|---|
| B3 backbone · 384px · batch 4 | ~4–6 GB ✅ | ~30s/epoch/100img |
| B3 backbone · 512px · batch 4 | ~7–9 GB ⚠️ | ~45s/epoch/100img |
| L2 backbone · 512px · batch 8 | >15 GB ❌ OOM | — |

> If you see **CUDA Out of Memory**: reduce batch size to 2 or image size to 224.

In [3]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 0 – GPU Check & Folder Setup                         ║
# ║  Run this FIRST every time you start or resume a session.  ║
# ╚══════════════════════════════════════════════════════════════╝

import os, sys, subprocess
import torch

print('=' * 60)
print('  USGS Water Level Prediction – Colab Training Demo')
print('=' * 60)

# ── GPU check ──────────────────────────────────────────────────
print('\n🖥️  GPU Status:')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'   ✅ GPU: {gpu_name}')
    print(f'   ✅ VRAM: {total_mem:.1f} GB')
    # Detailed memory summary
    print(f'   ✅ CUDA version: {torch.version.cuda}')
else:
    print('   ⚠️  No GPU detected!')
    print('   → Go to Runtime > Change runtime type > GPU > T4')

print(f'\n🐍 Python: {sys.version.split()[0]}')
print(f'🔦 PyTorch: {torch.__version__}')

# ── Folder structure ───────────────────────────────────────────
BASE_DIR = '/content/water_level_demo'
DIRS = [
    BASE_DIR,
    f'{BASE_DIR}/data',
    f'{BASE_DIR}/data/images',
    f'{BASE_DIR}/results',
]
for d in DIRS:
    os.makedirs(d, exist_ok=True)

print(f'\n📁 Folder structure created at: {BASE_DIR}')

# ── Optional: Mount Google Drive ───────────────────────────────
MOUNT_DRIVE = False  # ← Set to True if you want Drive integration

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('☁️  Google Drive mounted at /content/drive')
else:
    print('ℹ️   Google Drive NOT mounted (set MOUNT_DRIVE=True to enable)')

  USGS Water Level Prediction – Colab Training Demo

🖥️  GPU Status:
   ✅ GPU: NVIDIA A100-SXM4-80GB
   ✅ VRAM: 85.1 GB
   ✅ CUDA version: 12.8

🐍 Python: 3.12.13
🔦 PyTorch: 2.10.0+cu128

📁 Folder structure created at: /content/water_level_demo
ℹ️   Google Drive NOT mounted (set MOUNT_DRIVE=True to enable)


In [4]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 – Install & Verify Dependencies                    ║
# ║  This only needs to run once per Colab session.            ║
# ╚══════════════════════════════════════════════════════════════╝

import subprocess, sys

# Packages to install (torch / torchvision are pre-installed on Colab)
PACKAGES = [
    'timm>=0.9.0',
    'gradio>=4.0.0,<5.0.0',
    'pandas>=1.5.0',
    'numpy>=1.23.0',
    'scikit-learn>=1.1.0',
    'matplotlib>=3.6.0',
    'Pillow>=9.0.0',
    'tqdm>=4.64.0',
]

print('📦 Installing packages ...')
for pkg in PACKAGES:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        capture_output=True, text=True
    )
    status = '✅' if result.returncode == 0 else '❌'
    print(f'  {status} {pkg}')

# Verify key imports
print('\n🔍 Verifying imports ...')
import_checks = [
    ('torch',             'torch'),
    ('torchvision',       'torchvision'),
    ('timm',              'timm'),
    ('gradio',            'gradio'),
    ('sklearn',           'sklearn'),
    ('matplotlib',        'matplotlib'),
    ('PIL',               'PIL'),
]
all_ok = True
for display_name, mod_name in import_checks:
    try:
        mod = __import__(mod_name)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'  ✅ {display_name}: {ver}')
    except ImportError:
        print(f'  ❌ {display_name}: NOT FOUND – please install manually')
        all_ok = False

print()
if all_ok:
    print('✅ All dependencies are ready!')
else:
    print('⚠️  Some packages are missing. Check the errors above.')

📦 Installing packages ...
  ✅ timm>=0.9.0
  ✅ gradio>=4.0.0
  ✅ pandas>=1.5.0
  ✅ numpy>=1.23.0
  ✅ scikit-learn>=1.1.0
  ✅ matplotlib>=3.6.0
  ✅ Pillow>=9.0.0
  ✅ tqdm>=4.64.0

🔍 Verifying imports ...
  ✅ torch: 2.10.0+cu128
  ✅ torchvision: 0.25.0+cu128
  ✅ timm: 1.0.26
  ✅ gradio: 5.50.0
  ✅ sklearn: 1.6.1
  ✅ matplotlib: 3.10.0
  ✅ PIL: 11.3.0

✅ All dependencies are ready!


In [5]:
# QUICK FIX: Download train_demo.py and app.py from GitHub
import os, urllib.request, sys

BASE_DIR = '/content/water_level_demo'
os.makedirs(BASE_DIR, exist_ok=True)

# ── UPDATE this URL to your actual GitHub repo ──────────────────────────
GITHUB_RAW = 'https://raw.githubusercontent.com/yashsanap14/Water-prediction-colab-version/main'

for script in ['train_demo.py', 'data_acquisition.py', 'app.py']:
    dest = os.path.join(BASE_DIR, script)
    url  = f'{GITHUB_RAW}/{script}'
    try:
        urllib.request.urlretrieve(url, dest)
        size = os.path.getsize(dest)
        if size > 500:
            print(f'✅ Downloaded {script} ({size:,} bytes)')
        else:
            os.remove(dest)
            print(f'❌ {script}: got {size} bytes — URL may be wrong')
    except Exception as e:
        print(f'❌ {script}: {e}')

if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

import importlib, train_demo, app as gradio_app
importlib.reload(train_demo); importlib.reload(gradio_app)
print('\n✅ Imported successfully — now run Cell 3!')


✅ Downloaded train_demo.py (28,449 bytes)
✅ Downloaded app.py (23,091 bytes)

✅ Imported successfully — now run Cell 3!


In [6]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 – Launch Gradio Training UI                        ║
# ║                                                            ║
# ║  This launches the Gradio web interface.                   ║
# ║  A public share link will appear below – click it to open ║
# ║  the demo in your browser.                                 ║
# ╚══════════════════════════════════════════════════════════════╝

import sys
import importlib

BASE_DIR = '/content/water_level_demo'
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

# Reload modules in case code was updated
import train_demo
importlib.reload(train_demo)

import app as gradio_app
importlib.reload(gradio_app)

# GPU quick check
train_demo.check_gpu()

print('\n🌐 Launching Gradio UI ...')
print('   (share=True → a public link will appear below)')
print('   The link is valid for 72 hours.')
print()

# Launch the Gradio app
# share=True creates a public tunnelled URL (required for Colab)
demo = gradio_app.launch_gradio(share=True, debug=False)

✅ GPU detected: NVIDIA A100-SXM4-80GB  (85.1 GB VRAM)

🌐 Launching Gradio UI ...
   (share=True → a public link will appear below)
   The link is valid for 72 hours.



AttributeError: 'Textbox' object has no attribute 'page'

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 – (Optional) Run Training Directly from Python     ║
# ║                                                            ║
# ║  Use this if you prefer running training without the UI,   ║
# ║  or for debugging / scripted experiments.                  ║
# ╚══════════════════════════════════════════════════════════════╝

import sys, os
BASE_DIR = '/content/water_level_demo'
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

import train_demo as td
import pandas as pd

# ── Configuration – adjust these ─────────────────────────────
CSV_PATH   = f'{BASE_DIR}/data/labels.csv'         # path to your CSV
IMAGE_DIR  = f'{BASE_DIR}/data/images'             # path to your images folder
RESULTS    = f'{BASE_DIR}/results'

# Pinewood Road ROI (y1, x1, y2, x2)
ROI = (951, 0, 1136, 1920)

# ── Load and prepare data ────────────────────────────────────
df = pd.read_csv(CSV_PATH)
img_col, target_col, time_col = td.detect_columns(df)
print(f'Columns: image={img_col}, target={target_col}, time={time_col}')

mappings = td.build_image_label_mapping(
    df, img_col, target_col,
    image_dir=IMAGE_DIR,
    roi=None,
    max_images=200,
    seed=42,
)

# ── Train ────────────────────────────────────────────────────
results = td.train_model(
    mappings         = mappings,
    roi              = ROI,
    results_dir      = RESULTS,
    num_epochs       = 5,
    batch_size       = 4,
    input_img_size   = 384,
    learning_rate    = 2e-4,
    val_ratio        = 0.15,
    test_ratio       = 0.10,
    param_freeze_ratio = 0.7,
    seed             = 42,
    backbone_name    = 'tf_efficientnet_b3.ns_jft_in1k',
    save_to_drive    = False,
)

print('\nTraining complete!')
print('Best val loss:', results['best_val_loss'])
print('Outputs:', results)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 – View Training Results                            ║
# ╚══════════════════════════════════════════════════════════════╝

import os, json, pandas as pd
from IPython.display import display, Image as IPImage
import matplotlib.pyplot as plt

RESULTS = '/content/water_level_demo/results'

# ── List output files ─────────────────────────────────────────
print('📁 Files in results directory:')
if os.path.exists(RESULTS):
    for f in sorted(os.listdir(RESULTS)):
        size = os.path.getsize(os.path.join(RESULTS, f))
        print(f'  {f:40s}  {size:>10,} bytes')
else:
    print('  (empty – run training first)')

# ── Show config ───────────────────────────────────────────────
config_path = os.path.join(RESULTS, 'config.json')
if os.path.exists(config_path):
    print('\n📄 Training Config:')
    with open(config_path) as f:
        cfg = json.load(f)
    for k, v in cfg.items():
        print(f'  {k:25s}: {v}')

# ── Show loss history ─────────────────────────────────────────
history_path = os.path.join(RESULTS, 'training_history.csv')
if os.path.exists(history_path):
    print('\n📊 Loss History:')
    df = pd.read_csv(history_path)
    display(df)

# ── Display loss plot ─────────────────────────────────────────
plot_path = os.path.join(RESULTS, 'training_loss_plot.png')
if os.path.exists(plot_path):
    print('\n📈 Loss Plot:')
    display(IPImage(plot_path))

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 – Download Results ZIP                             ║
# ║                                                            ║
# ║  Creates a ZIP of all training outputs and downloads it.   ║
# ╚══════════════════════════════════════════════════════════════╝

import os, shutil
from google.colab import files

RESULTS   = '/content/water_level_demo/results'
ZIP_PATH  = '/content/water_level_training_outputs.zip'

if os.path.exists(RESULTS) and os.listdir(RESULTS):
    shutil.make_archive(
        base_name = ZIP_PATH.replace('.zip', ''),
        format    = 'zip',
        root_dir  = RESULTS,
    )
    print(f'✅ Created: {ZIP_PATH}')
    files.download(ZIP_PATH)
else:
    print('⚠️  No results to download. Run training first.')